# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

## 1. Setup

In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import json
from _campaign_lib import *

svc = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

Backend: http://127.0.0.1:8000


2026-03-17 16:28:22 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"
2026-03-17 16:28:22 INFO     [api.services.pipeline_discovery] Matched known pipeline 'termnorm'; using enriched schema
2026-03-17 16:28:22 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded: termnorm vv1.1


Pipeline: termnorm (6 steps)
Experiment: production_historical (40 queries, 93 session terms)
Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md


In [17]:
campaign_config = {
    "sample_size": 15,              # queries per eval step (service default: all)
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "exclude_steps": ["llm_ranking"],    # steps to skip (e.g. ["entity_profiling"])
    "pipeline_overrides": {},
    "optimization": {
        "patience": 2,                   # default: 3
        "max_rounds": 3,                 # default: 10
    },
    "eval_llm": {
        # --- Groq (free tier, open-source models) ---
        "model": "openai/gpt-oss-120b",
        # "model": "moonshotai/kimi-k2-instruct-0905"
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",          # best quality
        # "model": "claude-sonnet-4-6",      # good balance
        # "model": "claude-haiku-4-5-20251001",  # cheapest
        # "provider_url": "https://api.anthropic.com",
        "max_tokens": 2000,              # response length budget
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                    "descriptions to standardized database terms using entity profiling "
                    "and candidate ranking.",
        "grid_budget": 35,               # default: 0 (full grid)
        "sample_size": 6,     # default: 0 (all queries)
        "shared_queries": False,          # default: True
    },
}

In [18]:
#@title Pipeline snapshot (full config for reproducibility)
pipeline_config_full = await show_pipeline_snapshot(svc)

2026-03-17 16:28:22 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"


  PIPELINE SNAPSHOT: TermNorm v1.1
  Nodes:   ['fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking', 'direct_prompt']
  Schemas: ['entity_profile/1', 'llm_ranking_output/1']
  Prompts: ['entity_profiling/1', 'llm_ranking/1']

{
  "name": "TermNorm",
  "version": "v1.1",
  "available_models": [
    "moonshotai/kimi-k2-instruct-0905",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct",
    "openai/gpt-oss-120b"
  ],
  "nodes": {
    "fuzzy_matching": {
      "type": "DeterministicFunction",
      "config": {
        "threshold": 70,
        "scorer": "WRatio",
        "limit": 5
      }
    },
    "web_search": {
      "type": "ExternalService",
      "config": {
        "max_sites": 7,
        "num_results": 20,
        "content_char_limit": 800,
        "url_fetch_multiplier": 2,
        "fallback_keywords_limit": 8,
        "query_prefix": "",
        "query_suffix": "",
        "brave_api_timeout": 10,
        "scrape_tim

In [19]:
#@title Build pipeline params
pipeline_params = configure_pipeline(svc, campaign_config)

Active steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching']
  Excluded: ['llm_ranking']


## 2. Data

In [20]:
#@title Load datasets
# Set EXCEL_PATH to load from BOM-example.xlsx; leave empty to use stored data
EXCEL_PATH = r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx"  # e.g. "../data/BOM-example.xlsx"
FORCE_RELOAD = False  # Set True to re-read Excel and overwrite stored datasets

train_data, svc["session_terms"] = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=EXCEL_PATH or None,
    force=FORCE_RELOAD,
)


  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ------------------------------------------------
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets


In [21]:
#@title Prepare evaluation context
campaign_rounds = []
baseline_results = []

baseline, eval_data, backend_status = await prepare_eval_context(
    svc, train_data,
)

RUN_BASELINE = False  # Set True to evaluate baseline before exploration
if RUN_BASELINE:
    campaign_rounds, baseline_results = await run_baseline_eval(
        baseline, eval_data, campaign_config, svc,
    )

2026-03-17 16:28:22 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/status "HTTP/1.1 200 OK"



BACKEND STATUS
  Session Active                 True
  Active Sessions                1
  Terms Loaded                   89
  Match Database Identifiers     110
  Match Database Aliases         699
  Experiments Count              4
  Mappings Count                 1126
  Pipeline Version               v1.1
  Llm Provider                   groq
  Llm Model                      moonshotai/kimi-k2-instruct-0905
  ------------------------------------------------
  Experiments                   
    0_production_realtime        0 mappings
    1_production_historical      887 mappings
    2_bom_materials              159 mappings
    3_bom_processing             80 mappings

Evaluation data: 984 queries


In [22]:
#@title Experiment dashboard
# Set to a short hex ID (e.g. '68e2c5') to resume a specific experiment.
# The system adds prefixes (cycle_, scan_, etc.) per data type.
# Set to None to auto-detect from current campaign_config + eval_data.
EXPERIMENT_ID = '6c8f5ab73fcd' #None

# When EXPERIMENT_ID is set, load stored config → overrides notebook variables
if EXPERIMENT_ID:
    _stored_cfg = load_experiment_config(svc["store"], svc["backend_id"], EXPERIMENT_ID)
    if _stored_cfg:
        _stored_pp = _stored_cfg.get("pipeline_params")
        if _stored_pp and _stored_pp != (pipeline_params if "pipeline_params" in dir() else None):
            pipeline_params = _stored_pp
            campaign_config["pipeline_params"] = _stored_pp
            print(f"  Loaded pipeline_params from experiment {EXPERIMENT_ID}")

show_experiment_dashboard(
    svc["store"], svc["backend_id"],
    experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config,
    eval_data=eval_data,
    pipeline_params=pipeline_params if "pipeline_params" in dir() else None,
    baseline_prompt_state=campaign_rounds[0]["prompt_state"].model_dump() if campaign_rounds else None,
)

  Loaded pipeline_params from experiment 6c8f5ab73fcd

  EXPERIMENT: cycle_6c8f5ab73fcd
  Status: active  |  Rounds: 0  |  Best: 0.0%  |  Base: 10.0%
  Updated: 2026-03-17 09:05

  Config (copy to campaign_config to resume):
    max_rounds: 3
    patience: 2
    n_variants: 5
    creativity: 0.7
    improvement_threshold: 0.01
    model: openai/gpt-oss-120b
    temperature: 0.0
    sample_size: 15
    seed: 42
    pipeline_params: {max_token_candidates=30, profiling_schema=..., profiling_temperature=0.3, query_prefix=what material is, steps=...}


Diff: current config vs cycle_6c8f5ab73fcd
  (identical — will resume this campaign)
  → Config does NOT match — update campaign_config to resume



{'campaign_id': 'cycle_6c8f5ab73fcd',
 'created_at': '2026-03-17T09:05:09.124307+00:00',
 'updated_at': '2026-03-17T09:05:09.124307+00:00',
 'status': 'active',
 'n_trials': 0,
 'best_accuracy': 0.0,
 'best_trial_id': None,
 'baseline_accuracy': 0.1,
 'trials': [],
 'type': 'feedback_cycle',
 'config': {'max_rounds': 3,
  'patience': 2,
  'n_variants': 5,
  'creativity': 0.7,
  'improvement_threshold': 0.01,
  'model': 'openai/gpt-oss-120b',
  'provider': 'groq',
  'backend_url': 'http://127.0.0.1:8000',
  'backend_id': 'termnorm-local',
  'project_root': 'C:\\Users\\dsacc\\Desktop\\PromptPotter\\prompt-potter-optimizer\\.promptpotter\\projects',
  'generate_suggestions': False,
  'pipeline_params': {'steps': ['cache_lookup',
    'fuzzy_matching',
    'web_search',
    'entity_profiling',
    'token_matching'],
   'max_token_candidates': 30,
   'query_prefix': 'what material is',
   'profiling_schema': {'type': 'object',
    'properties': {'entity_name': {'type': 'string'},
     'core_

## 3. Explore

Two exploration paths: **Smart Search** (scan advisor + sensitivity scan) or **Grid Search** (brute-force sweep). Use one or both.

### 3a. Smart Search

In [23]:
#@title Browse variant library
display_variant_library()
# Filter examples:
# display_variant_library(source="PromptWizard")
# display_variant_library(axes=["thinking_style", "persona"])

Variant Library
  Sources: PromptPotter (16), PromptWizard (40)

  persona (6 variants — PromptPotter: 4, PromptWizard: 2)
    [ 0] [PromptPotter] (empty baseline)
    [ 1] [PromptPotter] You are a domain expert with deep knowledge of this field.
    [ 2] [PromptPotter] You are a precise, analytical system that evaluates candidates methodi...
    [ 3] [PromptPotter] You are a careful assistant that considers all options before deciding...
    [ 4] [PromptWizard 2024] You are a specialist who evaluates candidates based on domain-specific...
    [ 5] [PromptWizard 2024] You are a meticulous researcher who cross-references information befor...

  task_intent (4 variants — PromptPotter: 4)
    [ 0] [PromptPotter] (empty baseline)
    [ 1] [PromptPotter] Your task is to identify the single best match from the candidates.
    [ 2] [PromptPotter] Rank candidates by how well they match the concept described.
    [ 3] [PromptPotter] Evaluate each candidate for semantic equivalence to the query 

{'prompt_fields': {'persona': [{'text': '', 'source': 'PromptPotter'},
   {'text': 'You are a domain expert with deep knowledge of this field.',
    'source': 'PromptPotter'},
   {'text': 'You are a precise, analytical system that evaluates candidates methodically.',
    'source': 'PromptPotter'},
   {'text': 'You are a careful assistant that considers all options before deciding.',
    'source': 'PromptPotter'},
   {'text': 'You are a specialist who evaluates candidates based on domain-specific criteria and professional standards.',
    'source': 'PromptWizard',
    'year': 2024},
   {'text': 'You are a meticulous researcher who cross-references information before making judgments.',
    'source': 'PromptWizard',
    'year': 2024}],
  'task_intent': [{'text': '', 'source': 'PromptPotter'},
   {'text': 'Your task is to identify the single best match from the candidates.',
    'source': 'PromptPotter'},
   {'text': 'Rank candidates by how well they match the concept described.',
    'so

In [24]:
# preview_advisor_prompt()
preview_advisor_prompt(campaign_config, svc, task_description="TASK_DESCRIPTION", raw=True)

2026-03-17 16:28:22 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


You are an expert prompt optimization advisor. Recommend which axes (parameters and prompt fields) to prioritize in a sensitivity scan.

## Constraints (apply strictly)
- Do NOT recommend *_model axes — place them in axes_to_skip.
- Response must fit within 1500 tokens. Be terse.

## Pipeline: TermNorm AI terminology normalization pipeline
Steps execute sequentially — each step's output feeds the next:
[
  {
    "name": "cache_lookup",
    "node_role": "cache",
    "short_circuit": true
  },
  {
    "name": "fuzzy_matching",
    "node_role": "candidate_source",
    "short_circuit": true
  },
  {
    "name": "web_search",
    "node_role": "enricher"
  },
  {
    "name": "entity_profiling",
    "node_role": "enricher"
  },
  {
    "name": "token_matching",
    "node_role": "candidate_source"
  }
]

## Task Context
TASK_DESCRIPTION
## Tunable Parameters (per step)
[
  {
    "name": "fuzzy_matching",
    "param_keys": [
      "fuzzy_scorer",
      "fuzzy_threshold"
    ]
  },
  {
    "name

In [25]:
#@title Scan advisor
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc,
    task_description=TASK_DESCRIPTION if "TASK_DESCRIPTION" in dir() else "",
)

2026-03-17 16:28:22 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


SCAN ADVISOR -- pipeline-aware sensitivity setup
  Pipeline: termnorm (v1.1)
  Steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking']
  Excluded: ['llm_ranking']
  Task context: # Domain Context: Life Cycle Assessment (LCA) Terminology

This document capture...
  Calling openai/gpt-oss-120b ...



2026-03-17 16:28:27 INFO     [httpx] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


----------------------------------------------------------------------
PRIORITY AXES (ranked by importance)
----------------------------------------------------------------------
  1. [HIGH] fuzzy_threshold (pipeline_param) -- step: fuzzy_matching
     Controls match acceptance; tight thresholds reduce false positives
     Values: ['0.7', '0.85', '0.95']
  2. [MEDIUM] fuzzy_scorer (pipeline_param) -- step: fuzzy_matching
     Different scorers handle abbreviations vs numbers differently
     Values: ['ratio', 'partial_ratio', 'token_set_ratio']
  3. [HIGH] query_prefix (pipeline_param) -- step: web_search
     Guides search engine toward material context
     Values: ['', 'material', 'chemical']
  4. [HIGH] query_suffix (pipeline_param) -- step: web_search
     Adds domain‑specific qualifiers for better results
     Values: ['', 'datasheet', 'specifications']
  5. [MEDIUM] max_sites (pipeline_param) -- step: web_search
     Limits sites to balance coverage vs noise
     Values: ['5', '

In [26]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_sample_size = 10  # queries per scan variant (0 = use all)

scan_variants = {
    'max_token_candidates': [10, 30, 50],
    'query_prefix': ['what material is', 'identify LCA database name for', 'translate trade name'],
    'profiling_schema': [
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'], ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        # [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_synonyms', 'array', False, 'Terms likely to appear verbatim in LCA database entry names for this entity'], ['+', 'no_match_signal', 'string', False, 'Brief reasoning on whether a database match is likely to exist or not']],
        # [['~', 'classification_aliases', 'lca_classification_aliases', 'array', False, 'Expert-level aliases specifically aligned with LCA database naming conventions, including ecoinvent activity names and SimaPro process names'], ['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA']],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]], 
        [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]]
    ],
    'profiling_temperature': [0.0, 0.3, 0.7],
    # 'profiling_max_tokens': [512, 1024, 2048], # -> Going to cause lots of Errors.
    'raw_content_limit': [1000, 2500, 8000],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50]
  query_prefix: ['what material is', 'identify LCA database name for', 'translate trade name']
  profiling_schema: (baseline + 4 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'')
    [2] ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'')
    [3] ('-', 'manufacturing_processes'), ('-', 'applications'), ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions')
    [4] ('~', 'notes', 'material_category', 'string', True, 'The broad LCA material 

In [ ]:
#@title Prepare scan baseline
# Uses configure_pipeline() output (pre-seeding defaults), not campaign pipeline_params.
# The scan explores from defaults; seeded params are for the feedback cycle.
_scan_pp = configure_pipeline(svc, campaign_config)
baseline_sp, scan_coverage = await prepare_scan_baseline(
    baseline, campaign_config,
    pipeline_params=_scan_pp,
    store=svc["store"], backend_id=svc["backend_id"],
    scan_variants=scan_variants,
)

In [ ]:
#@title Sensitivity scan
scan_df, axis_profiles = await sensitivity_scan(
    baseline_sp, scan_variants, eval_data, svc.get("backend_client"),
    sample_size=scan_sample_size,
    store=svc["store"], backend_id=svc["backend_id"],
    pipeline_schema=svc.get("pipeline_schema"),
    experiment_id=EXPERIMENT_ID if "EXPERIMENT_ID" in dir() and EXPERIMENT_ID else "",
    scan_coverage=scan_coverage if "scan_coverage" in dir() else None,
)

In [ ]:
#@title Scan analytics: variant leaderboard
if scan_df is not None and not scan_df.empty:
    show_scan_leaderboard(scan_df, axis_profiles)

VARIANT LEADERBOARD (all scan combos)


,rank,axis,variant,accuracy,delta,hits/total,errors
0,1,max_token_candidates,30,30.0%,+18.0%,3/10,0
1,2,profiling_schema,schema(12 fields),20.0%,+9.2%,2/10,0
2,3,query_prefix,what material is,20.0%,+9.2%,2/10,0
3,4,profiling_temperature,0.3,20.0%,+9.0%,2/10,0
4,5,max_token_candidates,50,20.0%,+9.0%,2/10,0
5,6,raw_content_limit,8000,10.0%,+0.3%,1/10,0
6,7,profiling_schema,schema(11 fields),10.0%,+0.3%,1/10,0
7,8,profiling_schema,schema(11 fields),10.0%,+0.3%,1/10,0
8,9,raw_content_limit,1000,10.0%,+0.3%,1/10,0
9,10,profiling_temperature,0.7,10.0%,+0.3%,1/10,0



PER-AXIS STATISTICS


,axis,type,variants,mean_acc,std_acc,best_acc,worst_acc,sensitivity,budget
0,max_token_candidates,pipeline_param,3,16.7%,15.3%,30.0%,0.0%,0.270,medium
1,query_prefix,pipeline_param,3,6.7%,11.5%,20.0%,0.0%,0.185,medium
2,profiling_schema,pipeline_param,5,10.0%,7.1%,20.0%,0.0%,0.180,medium
3,profiling_temperature,pipeline_param,3,10.0%,10.0%,20.0%,0.0%,0.177,medium
4,raw_content_limit,pipeline_param,3,6.7%,5.8%,10.0%,0.0%,0.090,skip


In [ ]:
#@title Scan analytics: query difficulty
if scan_df is not None and not scan_df.empty:
    difficulty_df = show_scan_query_difficulty(
        svc["store"], svc["backend_id"],
    )

QUERY DIFFICULTY (728 queries across 59 scan runs)
  easy: 0 (0%) | discriminating: 5 (1%) | hard: 5 (1%) | error: 718 (99%)



,query,ground_truth,hit_rate,hits/evals,error_rate,classification
0,"BAND EN 10140-1,45x26 GK-DC01+C390-MB","Steel, unalloyed {GLO}| market for steel, unal...",0.000000,0/11,1.000000,error
1,"Kaltband EN 10140-2,5 x ... GK\nStahl EN 10139...","Steel removed by milling, small parts {RER}| s...",0.000000,0/11,1.000000,error
2,Adhesive label 13x5 white\nPolyethylen (PE) we...,"Polyethylene, low density, granulate {GLO}| ma...",0.000000,0/11,1.000000,error
3,"Strip EN13599-CU-PHC-R290-1,8x35-Ag0,3 /stamping","Metal working, average for copper product manu...",0.000000,0/22,1.000000,error
4,PA 66 25% GF V0 RAL 7012/0,Injection moulding {RoW}| injection moulding |...,0.000000,0/70,0.542857,hard
...,...,...,...,...,...,...
723,SJRG0013-PA/molding,Injection moulding {RER}| injection moulding |...,0.025000,1/40,0.550000,discriminating
724,SJRG0010-ABS/molding,Injection moulding {RER}| injection moulding |...,0.050847,3/59,0.457627,discriminating
725,Kingfa NPG25,Glass fibre reinforced plastic | 75% PA66 25% ...,0.220339,13/59,0.457627,discriminating
726,PA66-GF25 ULTRAMID A3UG5 RAL7035 grey,Glass fibre reinforced plastic | 75% PA66 25% ...,0.288136,17/59,0.457627,discriminating


In [ ]:
#@title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df, axis_profiles, baseline_sp, scan_variants,
    campaign_rounds, campaign_config,
)

2026-03-17 15:43:08 INFO     [api.services.search.scan_winner] select_scan_winner: 0 prompt changes, 4 param changes from 4 improving axes


Selected best from 4 improving axes:
  max_token_candidates      best_delta=+18.0%  value_idx=1  acc=30.0%
  query_prefix              best_delta=+9.2%  value_idx=0  acc=20.0%
  profiling_schema          best_delta=+9.2%  value_idx=2  acc=20.0%
  profiling_temperature     best_delta=+9.0%  value_idx=1  acc=20.0%
Pipeline params updated: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching'], 'max_token_candidates': 30, 'query_prefix': 'what material is', 'profiling_schema': {'type': 'object', 'properties': {'entity_name': {'type': 'string'}, 'core_concept': {'type': 'string', 'description': 'The single word that defines what this expression represents'}, 'distinguishing_features': {'type': 'array', 'items': {'type': 'string'}}, 'key_properties': {'type': 'array', 'items': {'type': 'string'}}, 'technical_specifications': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Explicit technical specs, dimensions, codes, ratings, tolerance

### 3b. Grid Search

<details>
<summary>Skip if you used Smart Search above.</summary>

Systematic sweep of the prompt configuration space. Maps the accuracy landscape before hill-climbing.

</details>

In [ ]:
# #@title Grid campaign overview (existing plans)
# merge_plans = False  # Set True to combine results from multiple plans
# grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
# merged_grid_df = grid_overview.get("merged_grid_df")

In [ ]:
# #@title Build or resume grid plan
# gs = campaign_config["grid_search"]

# llm_client, llm_model = setup_llm(campaign_config)

# (
#     grid_plan_id, grid_points, grid_state_lookup,
#     grid_axes, layer1_fields, grid_baseline,
# ) = await resume_or_build_grid(
#     campaign_config, baseline, llm_client, llm_model,
#     svc["store"], svc["backend_id"],
#     improvement_areas=campaign_config.get("improvement_areas", ""),
# )

# print(f"Grid points: {len(grid_points)}")
# print(f"Plan ID: {grid_plan_id}")

In [ ]:
# #@title Run grid search
# grid_df = await run_grid_search(
#     grid_points, grid_state_lookup, eval_data,
#     campaign_config["eval_llm"],
#     plan_id=grid_plan_id,
#     store=svc["store"], backend_id=svc["backend_id"],
#     backend_client=svc.get("backend_client"),
#     session_terms=svc.get("session_terms"),
#     pipeline_params=campaign_config.get("pipeline_params"),
#     sample_size=gs.get("sample_size", 1),
#     shared_queries=gs.get("shared_queries", False),
#     grid_seed=gs.get("seed", 42),
# )

In [ ]:
# #@title Display grid results
# _display_df = merged_grid_df if merged_grid_df is not None else grid_df
# display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
# #@title LLM analysis of grid results
# _analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
# llm_client, llm_model = setup_llm(campaign_config)
# grid_analysis = await analyze_grid_results(
#     _analysis_df, grid_axes, llm_client, model=llm_model,
# )

In [ ]:
# #@title Select grid winner and seed campaign
# grid_winner = select_and_seed_grid_winner(
#     grid_df, merged_grid_df, grid_state_lookup,
#     grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
# )

## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [ ]:
#@title Feedback cycle preflight
scan_context = show_feedback_preflight(
    campaign_rounds, eval_data, campaign_config,
    pipeline_params=pipeline_params,
    scan_df=scan_df if "scan_df" in dir() else None,
    axis_profiles=axis_profiles if "axis_profiles" in dir() else None,
    scan_variants=scan_variants if "scan_variants" in dir() else None,
    difficulty_df=difficulty_df if "difficulty_df" in dir() else None,
)

In [ ]:
#@title Run optimization (feedback cycle)
campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_url=svc["backend_client"].base_url,
    pipeline_params=pipeline_params,
    session_terms=svc.get("session_terms"),
    scan_context=scan_context if "scan_context" in dir() else None,
    experiment_id=EXPERIMENT_ID if "EXPERIMENT_ID" in dir() else None,
)

In [ ]:
#@title Run optimization round (manual)
round_entry = await run_manual_round(
    campaign_rounds, eval_data, campaign_config, svc,
    experiment_id=EXPERIMENT_ID if "EXPERIMENT_ID" in dir() else None,
)

## 5. Results

In [ ]:
#@title Campaign comparison table
show_campaign_summary(campaign_rounds)

In [ ]:
#@title Per-query flip tracking (baseline vs final)
show_flip_tracking(campaign_rounds)

In [ ]:
#@title PromptState lineage chain
show_lineage_chain(campaign_rounds)

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], svc["backend_id"])

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print("--- SUGGESTED CONFIG (copy to Setup) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

In [ ]:
#@title Sync evaluation history to Langfuse
# Safe to re-run — already-pushed runs are skipped automatically.
stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)